In [1]:
import pandas as pd
import numpy as np
import re
import glob

In [2]:
df = pd.read_csv("../raw_data/amazon_india_2025.csv")

In [139]:
df.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2025_00000001,2025-01-08,CUST_2025_00005600,PROD_000627,Oppo F11 Pro 128GB Black,Electronics,Smartphones,Oppo,10234.12,0.00,...,False,NaN,4.5,Delivered,1,2025,1,0.15,True,4.4
1,TXN_2025_00000002,01/15/2025,CUST_2022_00027099,PROD_001699,Samsung Slate 4GB RAM Silver,Electronics,Tablets,Samsung,38241.08,0.00,...,False,NaN,NaN,Returned,1,2025,1,0.64,TRUE,3.4
2,TXN_2025_00000003,2025-01-26,CUST_2021_00027917,PROD_001242,Apple iPhone 16 Plus 64GB Black,Electronics,Smartphones,Apple,121974.26,32.04,...,True,Republic Day Sale,NaN,Returned,1,2025,1,0.18,True,3.4
3,TXN_2025_00000004,2025-01-04,CUST_2025_00004184,PROD_000979,Samsung Galaxy S22+ 128GB White,Electronics,Smartphones,Samsung,59075.7,0.00,...,False,NaN,3.5,Delivered,1,2025,1,0.24,False,3.3
4,TXN_2025_00000005,2025-01-03,CUST_2025_00005205,PROD_001876,Apple Watch Premium,Electronics,Smart Watch,Apple,74269.31,0.00,...,False,NaN,5.0/5.0,Returned,1,2025,1,0.05,TRUE,4.1


In [140]:
df["delivery_charges"].isna().sum(), len(df)

(np.int64(6196), 77385)

In [141]:
df["delivery_charges"].describe()

count    71189.000000
mean         0.001124
std          0.212014
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         40.000000
Name: delivery_charges, dtype: float64

In [142]:
df.shape

(77385, 34)

In [143]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77385 entries, 0 to 77384
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_id          77385 non-null  object 
 1   order_date              77385 non-null  object 
 2   customer_id             77385 non-null  object 
 3   product_id              77385 non-null  object 
 4   product_name            77385 non-null  object 
 5   category                77385 non-null  object 
 6   subcategory             77385 non-null  object 
 7   brand                   77385 non-null  object 
 8   original_price_inr      77385 non-null  object 
 9   discount_percent        77385 non-null  float64
 10  discounted_price_inr    77385 non-null  float64
 11  quantity                77385 non-null  int64  
 12  subtotal_inr            77385 non-null  float64
 13  delivery_charges        71189 non-null  float64
 14  final_amount_inr        77385 non-null

In [144]:
df.columns

Index(['transaction_id', 'order_date', 'customer_id', 'product_id',
       'product_name', 'category', 'subcategory', 'brand',
       'original_price_inr', 'discount_percent', 'discounted_price_inr',
       'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr',
       'customer_city', 'customer_state', 'customer_tier',
       'customer_spending_tier', 'customer_age_group', 'payment_method',
       'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale',
       'festival_name', 'customer_rating', 'return_status', 'order_month',
       'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible',
       'product_rating'],
      dtype='object')

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [145]:
df["order_date"].head(20)

0     2025-01-08
1     01/15/2025
2     2025-01-26
3     2025-01-04
4     2025-01-03
5     2025-01-07
6     2025-01-09
7     26-01-2025
8     2025-01-28
9     2025-01-22
10    2025-01-10
11    2025-01-22
12    2025-01-29
13    2025-01-27
14    2025-01-01
15    2025-01-25
16    2025-01-20
17    2025-01-26
18    2025-01-27
19    2025-01-31
Name: order_date, dtype: object

In [146]:
df["order_date"] = (
    df["order_date"]
    .str.replace(" ", "", regex=False)
    .str.replace("/", "-", regex=False)
)

parts = df["order_date"].str.split("-", expand=True)

year_last = parts[2].str.len() == 4

df.loc[year_last, "order_date"] = (
    parts[2] + "-" + parts[0] + "-" + parts[1]
)

parts = df["order_date"].str.split("-", expand=True)

mask = parts[1].astype(int) > 12

df.loc[mask, "order_date"] = (
    parts[0] + "-" + parts[2] + "-" + parts[1]
)

df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

In [147]:
df["order_date"].min(), df["order_date"].max()

(Timestamp('2025-01-01 00:00:00'), Timestamp('2025-12-31 00:00:00'))

In [148]:
df["order_date"].isna().sum()

np.int64(0)

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees. 


In [149]:
df["original_price_inr"] = df["original_price_inr"].astype(str)

df["original_price_inr"] = df["original_price_inr"].str.replace("₹", "", regex=False)

df["original_price_inr"] = df["original_price_inr"].str.replace(",", "", regex=False)

df["original_price_inr"] = pd.to_numeric(df["original_price_inr"], errors="coerce")

In [150]:
df["original_price_inr"].unique()[:20]

array([ 10234.12,  38241.08, 121974.26,  59075.7 ,  74269.31,  42119.59,
       128154.25,  19316.59,  97356.55,       nan,  24841.29,  62085.4 ,
        47195.75,  51649.26, 102078.41,  25292.65,  24121.43,  10167.72,
        21769.59,   6857.33])

In [151]:
mask = df["original_price_inr"].isna()

df.loc[mask, "original_price_inr"] = np.where(
    df.loc[mask, "discount_percent"] == 0,
    
    # Case 1: no discount
    df.loc[mask, "discounted_price_inr"],
    
    # Case 2: discount present
    df.loc[mask, "discounted_price_inr"] / (1 - df.loc[mask, "discount_percent"] / 100)
)

In [152]:
df["original_price_inr"].dtypes

dtype('float64')

In [153]:
df["original_price_inr"].isna().sum()

np.int64(0)

Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.


In [154]:
df["customer_rating"] = df["customer_rating"].astype(str)

df["customer_rating"] = df["customer_rating"].str.replace(" stars", "", regex=False)

df["customer_rating"] = df["customer_rating"].str.split("/").str[0]

df["customer_rating"] = pd.to_numeric(df["customer_rating"], errors="coerce")

In [155]:
df["customer_rating"].describe()

count    53921.000000
mean         4.305744
std          0.570673
min          3.000000
25%          4.000000
50%          4.500000
75%          5.000000
max          5.000000
Name: customer_rating, dtype: float64

In [156]:
df["customer_rating"].value_counts().head(10)

customer_rating
4.5    17732
4.0    13898
5.0    13581
3.5     5498
3.0     3212
Name: count, dtype: int64

In [157]:
df["customer_rating"].isna().sum()

np.int64(23464)

In [158]:
df["customer_rating"].value_counts(dropna=False)

customer_rating
NaN    23464
4.5    17732
4.0    13898
5.0    13581
3.5     5498
3.0     3212
Name: count, dtype: int64

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.


In [159]:
df["customer_city"] = df["customer_city"].str.strip().str.lower()

In [160]:
df["customer_city"].unique()

array(['delhi', 'hyderabad', 'bangalore', 'chandigarh', 'kolkata', 'pune',
       'ahmedabad', 'bhubaneswar', 'chennai', 'surat', 'vadodara',
       'nagpur', 'ludhiana', 'visakhapatnam', 'indore', 'jaipur',
       'kanpur', 'bareilly', 'aligarh', 'mumbai', 'coimbatore', 'patna',
       'moradabad', 'gorakhpur', 'lucknow', 'saharanpur', 'kochi',
       'allahabad', 'varanasi', 'meerut', 'banglore', 'madras',
       'calcutta', 'chenai', 'mumba', 'new delhi', 'delhi ncr', 'bombay',
       'bengaluru', 'bengalore'], dtype=object)

In [161]:
df["customer_city"] = df["customer_city"].str.strip().str.lower()

In [162]:
city_map = {
    "new delhi": "delhi",
    "delhi ncr": "delhi",

    "bombay": "mumbai",
    "mumba": "mumbai",

    "madras": "chennai",
    "chenai": "chennai",

    "calcutta": "kolkata",

    "bengaluru": "bangalore",
    "banglore": "bangalore",
    "bengalore": "bangalore"
}

In [163]:
df["customer_city"] = df["customer_city"].replace(city_map)

In [164]:
df["customer_city"] = df["customer_city"].str.title()

In [165]:
df["customer_city"].value_counts().head(20)

customer_city
Mumbai           7264
Delhi            6597
Bangalore        5637
Pune             5220
Chennai          4621
Ahmedabad        3843
Kolkata          3669
Jaipur           3189
Surat            3027
Nagpur           3014
Kanpur           2822
Indore           2763
Lucknow          2599
Coimbatore       2309
Hyderabad        2286
Kochi            2127
Visakhapatnam    1962
Patna            1884
Vadodara         1841
Bhubaneswar      1815
Name: count, dtype: int64

In [166]:
df["customer_city"].unique()

array(['Delhi', 'Hyderabad', 'Bangalore', 'Chandigarh', 'Kolkata', 'Pune',
       'Ahmedabad', 'Bhubaneswar', 'Chennai', 'Surat', 'Vadodara',
       'Nagpur', 'Ludhiana', 'Visakhapatnam', 'Indore', 'Jaipur',
       'Kanpur', 'Bareilly', 'Aligarh', 'Mumbai', 'Coimbatore', 'Patna',
       'Moradabad', 'Gorakhpur', 'Lucknow', 'Saharanpur', 'Kochi',
       'Allahabad', 'Varanasi', 'Meerut'], dtype=object)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [167]:
bool_candidates = []

bool_values = {"true","false","yes","no","y","n","1","0"}

for col in df.columns:
    vals = set(df[col].astype(str).str.lower().dropna().unique())
    
    if vals & bool_values:
        bool_candidates.append(col)

bool_candidates

['quantity',
 'delivery_days',
 'is_prime_member',
 'is_festival_sale',
 'order_month',
 'order_quarter',
 'is_prime_eligible']

In [168]:
boolean_cols = ["is_prime_member", "is_prime_eligible", "is_festival_sale"]

bool_map = {
    "true": True,
    "false": False,
    "yes": True,
    "no": False,
    "y": True,
    "n": False,
    "1": True,
    "0": False
}

for col in boolean_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(bool_map)
    )

In [169]:
df[boolean_cols].value_counts(dropna=False)

is_prime_member  is_prime_eligible  is_festival_sale
True             True               False               28480
False            True               False               16023
True             True               True                12580
False            True               True                 6950
True             False              False                6069
False            False              False                3178
True             False              True                 2671
False            False              True                 1434
Name: count, dtype: int64

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.


In [3]:
df["category"].value_counts().head(20)

category
Electronics                  77303
Electronicss                    25
ELECTRONICS                     22
Electronics & Accessories       20
Electronic                      15
Name: count, dtype: int64

In [4]:
category_map = {
    "ELECTRONICS": "Electronics",
    "Electronics & Accessories": "Electronics",
    "Electronic": "Electronics",
    "Electronicss": "Electronics"
}

In [5]:
df["category"] = df["category"].replace(category_map)
df["category"] = df["category"].str.title()

In [6]:
df["category"].value_counts()

category
Electronics    77385
Name: count, dtype: int64

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [174]:
df["delivery_days"].unique()

array(['3', '2', '1', '5', '6', '4', '0', '-1', '1-2 days', '7',
       'Same Day', 'Express', '15'], dtype=object)

In [175]:
df["delivery_days"] = df["delivery_days"].astype(str).str.strip().str.lower()

In [176]:
df["delivery_days"] = df["delivery_days"].replace({
    "same day": "0",
    "express": "1"
})

In [177]:
df["delivery_days"] = df["delivery_days"].str.extract(r"(-?\d+)")

In [178]:
df["delivery_days"] = pd.to_numeric(df["delivery_days"], errors="coerce")

In [179]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = None

In [180]:
df["delivery_days"].unique()

array([ 3.,  2.,  1.,  5.,  6.,  4.,  0., nan,  7., 15.])

In [181]:
df["delivery_days"].isnull().sum()

np.int64(489)

In [182]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = np.nan

df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median())

In [183]:
df["delivery_days"].describe()

count    77385.000000
mean         2.637165
std          1.674251
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         15.000000
Name: delivery_days, dtype: float64

In [184]:
df["delivery_days"].isna().sum()

np.int64(0)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [185]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [186]:
duplicates.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
480,TXN_2025_00000481,2025-01-19,CUST_2025_00008852,PROD_000271,OnePlus OnePlus 5T 32GB Black,Electronics,Smartphones,OnePlus,25617.50,0.00,...,False,NaN,4.5,Delivered,1,2025,1,0.18,False,4.1
657,TXN_2025_00000658,2025-01-02,CUST_2023_00004475,PROD_001056,Nothing Phone (1) 256GB Black,Electronics,Smartphones,Nothing,36543.46,0.00,...,False,NaN,3.0,Delivered,1,2025,1,0.15,True,4.3
723,TXN_2025_00000724,2025-01-28,CUST_2025_00014658,PROD_000751,Realme Realme Narzo 10 64GB Black,Electronics,Smartphones,Realme,28240.47,0.00,...,False,NaN,NaN,Delivered,1,2025,1,0.19,False,4.3
787,TXN_2025_00000788,2025-01-25,CUST_2024_00019334,PROD_001986,Samsung QLED TV,Electronics,TV & Entertainment,Samsung,25453.30,54.27,...,True,Republic Day Sale,4.0,Delivered,1,2025,1,44.26,True,3.7
1125,TXN_2025_00001126,2025-01-18,CUST_2019_00002863,PROD_001149,Xiaomi Mi 13 256GB Black,Electronics,Smartphones,Xiaomi,21450.82,0.00,...,False,NaN,5.0,Delivered,1,2025,1,0.17,True,3.4


In [187]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [188]:
duplicates.shape

(758, 34)

In [189]:
df.duplicated().sum()

np.int64(0)

In [190]:
duplicates.sort_values(
    ["customer_id","product_id","order_date"]
).head(10)

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
66774,TXN_2025_00066775,2025-11-14,CUST_2015_00001642,PROD_000002,Apple iPhone 6 32GB Black,Electronics,Smartphones,Apple,120522.07,14.08,...,False,NaN,NaN,Delivered,11,2025,4,0.18,True,3.4
77259,TXN_2025_00066775_DUP,2025-11-14,CUST_2015_00001642,PROD_000002,Apple iPhone 6 32GB Black,Electronics,Smartphones,Apple,120522.07,14.08,...,False,NaN,NaN,Delivered,11,2025,4,0.18,True,3.4
74631,TXN_2025_00074632,2025-12-16,CUST_2015_00010066,PROD_001958,Noise Sports Watch Deluxe,Electronics,Smart Watch,Noise,33216.07,0.00,...,False,NaN,3.0,Delivered,12,2025,4,0.06,True,3.6
77173,TXN_2025_00074632_DUP,2025-12-16,CUST_2015_00010066,PROD_001958,Noise Sports Watch Deluxe,Electronics,Smart Watch,Noise,33216.07,0.00,...,False,NaN,3.0,Delivered,12,2025,4,0.06,True,3.6
55963,TXN_2025_00055964,2025-10-07,CUST_2016_00002404,PROD_000262,OnePlus OnePlus 5 16GB Black,Electronics,Smartphones,OnePlus,27337.72,0.00,...,False,NaN,4.5,Delivered,10,2025,4,0.20,True,4.6
77178,TXN_2025_00055964_DUP,2025-10-07,CUST_2016_00002404,PROD_000262,OnePlus OnePlus 5 16GB Black,Electronics,Smartphones,OnePlus,27337.72,0.00,...,False,NaN,4.5,Delivered,10,2025,4,0.20,True,4.6
59900,TXN_2025_00059901,2025-10-15,CUST_2017_00014135,PROD_001904,Xiaomi Tracker Premium,Electronics,Smart Watch,Xiaomi,49950.70,46.98,...,True,Diwali Sale,5.0,Delivered,10,2025,4,0.06,False,4.1
77290,TXN_2025_00059901_DUP,2025-10-15,CUST_2017_00014135,PROD_001904,Xiaomi Tracker Premium,Electronics,Smart Watch,Xiaomi,49950.70,46.98,...,True,Diwali Sale,5.0,Delivered,10,2025,4,0.06,False,4.1
31809,TXN_2025_00031810,2025-06-23,CUST_2017_00018732,PROD_000655,Apple iPhone 12 Pro Max 128GB Blue,Electronics,Smartphones,Apple,133843.96,42.78,...,True,Back to School,4.5,Delivered,6,2025,2,0.17,True,3.7
77109,TXN_2025_00031810_DUP,2025-06-23,CUST_2017_00018732,PROD_000655,Apple iPhone 12 Pro Max 128GB Blue,Electronics,Smartphones,Apple,133843.96,42.78,...,True,Back to School,4.5,Delivered,6,2025,2,0.17,True,3.7


In [191]:
duplicates.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().sort_values(ascending=False).head(10)

customer_id         product_id   order_date  original_price_inr
CUST_2015_00001642  PROD_000002  2025-11-14  120522.07             2
CUST_2015_00010066  PROD_001958  2025-12-16  33216.07              2
CUST_2016_00002404  PROD_000262  2025-10-07  27337.72              2
CUST_2017_00014135  PROD_001904  2025-10-15  49950.70              2
CUST_2017_00018732  PROD_000655  2025-06-23  133843.96             2
CUST_2017_00020409  PROD_000730  2025-11-21  25595.62              2
CUST_2017_00023959  PROD_000392  2025-03-07  56389.37              2
CUST_2017_00026296  PROD_000684  2025-05-04  78267.94              2
CUST_2018_00002992  PROD_000921  2025-10-25  19450.12              2
CUST_2018_00025733  PROD_001742  2025-12-18  37349.84              2
dtype: int64

In [192]:
dup_groups = df.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().reset_index(name="count")

dup_groups = dup_groups[dup_groups["count"] > 1]

In [193]:
dup_rows = df.merge(
    dup_groups,
    on=["customer_id","product_id","order_date","original_price_inr"],
    how="inner"
)

In [194]:
dup_rows[["customer_id","product_id","quantity","count"]].head()

,customer_id,product_id,quantity,count
0,CUST_2025_00008852,PROD_000271,2,2
1,CUST_2023_00004475,PROD_001056,1,2
2,CUST_2025_00014658,PROD_000751,1,2
3,CUST_2024_00019334,PROD_001986,1,2
4,CUST_2019_00002863,PROD_001149,1,2


In [195]:
df_clean = df.drop_duplicates(
    subset=["customer_id","product_id","order_date","original_price_inr"],
    keep="first"
)

In [196]:
df_clean.duplicated(
    subset=["customer_id","product_id","order_date","original_price_inr"]
).sum()

np.int64(0)

In [197]:
df["transaction_id"].duplicated().sum()

np.int64(0)

In [198]:
df = df.drop_duplicates(subset="transaction_id", keep="first")

In [199]:
df[df["transaction_id"]=="TXN_2015_00000280"]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating


Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [200]:

# ── FIX 1: Negative prices ──────────────────────────
neg_mask = df["original_price_inr"] < 0
df.loc[neg_mask, "original_price_inr"] = df.loc[neg_mask, "original_price_inr"].abs()
print(f"Negative prices fixed: {neg_mask.sum()}")

# ── FIX 2: Outliers ──────────────────────────────────
subcategory_caps = {
    "Smart Watch":        100000,
    "Tablets":            180000,
    "Smartphones":        400000,
    "Laptops":            550000,
    "TV & Entertainment": 500000,
    "Audio":              200000,
}

outlier_mask = df.apply(
    lambda row: row["original_price_inr"] > subcategory_caps.get(row["subcategory"], 999999),
    axis=1
)
df.loc[outlier_mask, "original_price_inr"] = (
    df.loc[outlier_mask, "original_price_inr"] / 100
).round(2)
print(f"Outliers fixed: {outlier_mask.sum()}")

# ── FIX 3: delivery_charges ──────────────────────────
df["delivery_charges"] = df["delivery_charges"].fillna(0)

# ── FIX 4: Recalculate ───────────────────────────────
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = (df["subtotal_inr"] + df["delivery_charges"]).round(2)

# ── VERIFY ───────────────────────────────────────────
print(df.groupby("subcategory", observed=True)["original_price_inr"]
      .describe()[["min","max","mean","50%"]].round(2))
print(f"\nNaN in final_amount_inr:   {df['final_amount_inr'].isna().sum()}")
print(f"Negative prices remaining: {(df['original_price_inr'] < 0).sum()}")


Negative prices fixed: 206
Outliers fixed: 309
                        min        max      mean       50%
subcategory                                               
Audio               1067.27  193836.30  20404.16  21970.81
Laptops             9637.34  446284.80  86107.52  73503.23
Smart Watch         2049.05   74269.31  38497.00  36865.44
Smartphones         4041.82  396216.80  46173.32  28282.91
TV & Entertainment  6477.14  150188.32  72212.68  78446.90
Tablets             1875.62  178718.01  76109.42  70804.42

NaN in final_amount_inr:   0
Negative prices remaining: 0


In [201]:
# Check if any legitimate products were over-corrected in cleaned files
for sub, cap in subcategory_caps.items():
    over = df[(df["subcategory"] == sub) & 
                    (df["original_price_inr"] > cap)]
    if len(over) > 0:
        print(f"\n{sub} (cap ₹{cap:,}): {len(over)} rows over")
        print(over[["product_name", "original_price_inr"]].drop_duplicates().head(5))
    else:
        print(f"\n{sub}: ✅ All within cap")


Smart Watch: ✅ All within cap

Tablets: ✅ All within cap

Smartphones: ✅ All within cap

Laptops: ✅ All within cap

TV & Entertainment: ✅ All within cap

Audio: ✅ All within cap


Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 
'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [202]:
# ── Standardize payment methods ────────────────────
payment_standardize = {
    # UPI variants
    "UPI": "UPI", "PhonePe": "UPI", "GooglePay": "UPI", "Google Pay": "UPI",
    "UPI/PhonePe": "UPI", "UPI/GooglePay": "UPI",

    # Credit Card variants
    "Credit Card": "Credit Card", "CREDIT_CARD": "Credit Card", "CC": "Credit Card",

    # Debit Card variants
    "Debit Card": "Debit Card", "DEBIT_CARD": "Debit Card", "DC": "Debit Card",

    # COD variants
    "Cash on Delivery": "COD", "COD": "COD", "C.O.D": "COD",

    # Others
    "Wallet": "Wallet",
    "Net Banking": "Net Banking",
    "BNPL": "BNPL"
}

df["payment_method"] = df["payment_method"].map(payment_standardize).fillna(df["payment_method"])

# ── Create categorical hierarchy ───────────────────
payment_category = {
    "UPI":          "Digital Payment",
    "Wallet":       "Digital Payment",
    "Net Banking":  "Digital Payment",
    "Credit Card":  "Card Payment",
    "Debit Card":   "Card Payment",
    "BNPL":         "Pay Later",
    "COD":          "Cash on Delivery"
}

df["payment_category"] = df["payment_method"].map(payment_category).astype("category")

# ── Verify ─────────────────────────────────────────
print(df["payment_method"].value_counts())
print(f"\nNaN in payment_category: {df['payment_category'].isna().sum()}")

payment_method
UPI            46539
Credit Card     9286
COD             6236
Debit Card      6225
BNPL            5266
Net Banking     2285
Wallet          1548
Name: count, dtype: int64

NaN in payment_category: 0


Handling Nan - in customer age group

In [203]:
df["customer_age_group"].dtype

dtype('O')

In [204]:
df["customer_age_group"] = df["customer_age_group"].cat.add_categories(["Unknown"])
df["customer_age_group"] = df["customer_age_group"].fillna("Unknown")

AttributeError: Can only use .cat accessor with a 'category' dtype

In [ ]:
df["customer_age_group"].cat.categories

Index(['18-25', '26-35', '36-45', '46-55', '55+', 'Unknown'], dtype='object')

Checking for object columns

In [ ]:
df.select_dtypes(include="object").columns

Index(['transaction_id', 'customer_id', 'product_id', 'product_name', 'brand',
       'customer_city', 'customer_state'],
      dtype='object')

In [ ]:
# ── Optimize memory: convert to category dtype ───────
cat_columns = ["category", "subcategory", "customer_tier", 
               "customer_spending_tier", "customer_age_group",
               "payment_method", "delivery_type", 
               "festival_name", "return_status"]

df[cat_columns] = df[cat_columns].astype("category")

# Verify
print(df[cat_columns].dtypes)
print(f"\nMemory usage after optimization:")
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

category                  category
subcategory               category
customer_tier             category
customer_spending_tier    category
customer_age_group        category
payment_method            category
delivery_type             category
festival_name             category
return_status             category
dtype: object

Memory usage after optimization:
42.19951248168945 MB


In [205]:
df.to_csv("data_cleaning_2025.csv", index=False)
print("File saved successfully!")

File saved successfully!
